# Estudo de Caso 4: Eficiência da Conciliação no 4º JEC de Natal

**Arquivos de entrada**
- `dat/estcaso_4JEC_natal/ControleDosProcessos_Mestrado_Atualizacao_07.10.2026.xlsx`;
- `txt/24_09-TrabalhoEmpirico-PauloGiovani-Enajus2026-FINAL-cleuler-corrigido.docx`;
- `txt/limitacoes_experimento_4_JEC_Natal.odt`.

Este notebook foi estruturado como extensão do `estcaso_ip.ipynb`, com organização metodológica alinhada ao fluxo analítico de `rafaelcc_ed1.ipynb`. Assim, busca-se preservar a reprodutibilidade, a rastreabilidade das transformações e a consistência inferencial ao longo de todo o pipeline.


## Introdução

Este estudo examina o tempo de tramitação processual no 4º Juizado Especial Cível de Natal, com comparação entre grupos de rito e modalidades de conciliação. Para tanto, a análise integra três fontes complementares:

1. **Base processual** (planilha XLSX);
2. **Contexto teórico e delineamento empírico** (DOCX);
3. **Limitações metodológicas do experimento** (ODT).

A variável principal é o **tempo de tramitação** (em dias), com atenção específica para:
- a censura à direita (processos ativos);
- as diferenças entre grupos experimentais;
- a robustez inferencial, por meio de testes não paramétricos e análise de sobrevivência.


## 1. Tipos de Gráficos: Introdução Visual

| Gráfico | Tipo de variável | Finalidade analítica |
|---|---|---|
| Barras / Pareto | Categórica | Avaliar frequências e concentração de categorias. |
| Histograma / KDE | Quantitativa | Examinar forma, dispersão e assimetria da distribuição. |
| Boxplot / Violin | Quantitativa por grupos | Comparar posição, dispersão e valores extremos. |
| Mosaico | Categórica × categórica | Investigar associação entre categorias. |
| Dispersão + Regressão | Quantitativa × quantitativa | Identificar tendência e intensidade da relação. |
| Kaplan-Meier | Tempo até evento | Modelar tramitação com censura. |


## 2. Preparação do Ambiente


**Frase-guia didatica.** Nesta etapa, executaremos a rotina de **configuracao inicial do ambiente**.

> **Atencao a interpretacao:** ao analisar os resultados, confirme se todas as bibliotecas foram carregadas sem erro e se os caminhos de entrada/saida foram reconhecidos.

In [1]:
# Passo a passo da celula:
# 1) Importar bibliotecas e dependencias necessarias.
# 2) Definir caminhos de entrada e saida usados no estudo.
# 3) Ler fontes de dados/textos para preparacao analitica.
# 4) Produzir graficos para visualizacao dos padroes.
# 5) Aplicar analise de sobrevivencia com dados censurados.
# 6) Exibir saidas intermediarias para validacao da etapa.

# Bibliotecas principais de manipulação, estatística e visualização
from pathlib import Path
import warnings
import re
import unicodedata
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Bibliotecas opcionais (se faltarem, o notebook segue nas partes essenciais)
try:
    import statsmodels.api as sm
    from statsmodels.graphics.mosaicplot import mosaic
except Exception:
    sm = None
    mosaic = None

try:
    from lifelines import KaplanMeierFitter, CoxPHFitter
except Exception:
    KaplanMeierFitter = None
    CoxPHFitter = None

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_context("notebook")

BASE = Path.cwd()
DADOS_XLSX = BASE / "dat" / "estcaso_4JEC_natal" / "ControleDosProcessos_Mestrado_Atualizacao_07.10.2026.xlsx"
TEXTO_DOCX = BASE / "txt" / "24_09-TrabalhoEmpirico-PauloGiovani-Enajus2026-FINAL-cleuler-corrigido.docx"
TEXTO_ODT = BASE / "txt" / "limitacoes_experimento_4_JEC_Natal.odt"
SAIDA_DIR = BASE / "dat" / "processado"
SAIDA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base de trabalho: {BASE}")
print(f"Planilha encontrada? {DADOS_XLSX.exists()}")
print(f"DOCX encontrado? {TEXTO_DOCX.exists()}")
print(f"ODT encontrado? {TEXTO_ODT.exists()}")

Base de trabalho: e:\DADOS\Documents\AmbienteDeTrabalho\Projetos\notebook
Planilha encontrada? True
DOCX encontrado? True
ODT encontrado? True


## 3. Criação do Dataset


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **extração e normalização dos textos de suporte metodológico**.

> **Atenção e interpretação:** ao analisar os resultados, observe se os trechos impressos estão legíveis e coerentes com o contexto do estudo.

In [2]:
# Passo a passo da célula:
# 1) Declarar funções auxiliares para organizar a lógica.
# 2) Ler fontes de dados/textos para preparação analítica.
# 3) Exibir saídas intermediárias para validação da etapa.

# Funções auxiliares para leitura dos textos metodológicos

def extrair_texto_docx(caminho: Path) -> str:
    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
    with zipfile.ZipFile(caminho) as z:
        root = ET.fromstring(z.read("word/document.xml"))
    paragrafos = []
    for p in root.findall(".//w:p", ns):
        partes = [t.text for t in p.findall(".//w:t", ns) if t.text]
        if partes:
            paragrafos.append("".join(partes))
    return "\n".join(paragrafos)

# Finalidade: extrair e concatenar o texto dos parágrafos do arquivo DOCX em uma única string.


def extrair_texto_odt(caminho: Path) -> str:
    with zipfile.ZipFile(caminho) as z:
        root = ET.fromstring(z.read("content.xml"))
    trechos = [el.text.strip() for el in root.iter() if el.text and el.text.strip()]
    return "\n".join(trechos)

# Finalidade: extrair os trechos textuais relevantes do ODT e retornar conteúdo linearizado.


def normalizar_texto_curto(txt: str) -> str:
    txt = re.sub(r"\s+", " ", txt).strip()
    return txt

# Finalidade: reduzir ruído textual (múltiplos espaços/quebras) para facilitar a leitura dos excertos.


texto_docx = extrair_texto_docx(TEXTO_DOCX)
texto_odt = extrair_texto_odt(TEXTO_ODT)

print("Trecho DOCX:")
print(normalizar_texto_curto(texto_docx)[:700], "...\n")
print("Trecho ODT:")
print(normalizar_texto_curto(texto_odt)[:700], "...")


Trecho DOCX:
Eficácia da Conciliação Obrigatória na Tramitação Processual: um Experimento no 4º Juizado Especial Cível de Natal (2026) Paulo Giovani Militão de Alencar – Universidade Federal de Goiás Cleuler Barbosa das Neves – Universidade Federal de Goiás Tema de interesse: Métodos alternativos de solução de demandas judiciais RESUMO O presente trabalho empírico busca avaliar a conciliação como política pública levada a efeito nacionalmente pelo Conselho Nacional de Justiça, por meio da Resolução nº 125/2010, bem como, em nível local, pelo Tribunal de Justiça do Rio Grande do Norte. A conciliação surge como esperança no enfrentamento da morosidade judicial na tramitação dos processos, despontando legal ...

Trecho ODT:
D A seleção de um único juizado (4º JEC) por conveniência e a premissa de homogeneidade entre as unidades judiciárias, somadas à estrutura temporal adotada, introduzem severas limitações metodológicas à validade interna e externa do experimento. As principais limitaçõe

**Frase-guia didática.** Nesta etapa, executaremos a rotina de **leitura da planilha-base e inspeção estrutural**.

> **Atenção e interpretação:** ao analisar os resultados, verifique dimensões, nomes de colunas e presença de valores faltantes já na leitura inicial.

In [3]:
# Passo a passo da celula:
# 1) Ler fontes de dados/textos para preparacao analitica.
# 2) Converter tipos para formato analitico consistente.
# 3) Exibir saidas intermediarias para validacao da etapa.

# Leitura da planilha principal (aba 'Processos') preservando cabeçalho real na linha 2
raw = pd.read_excel(DADOS_XLSX, sheet_name="Processos", header=None)

# A primeira linha útil de cabeçalho está no índice 1
cabecalho_original = raw.iloc[1].fillna("").astype(str).str.strip()
dados_brutos = raw.iloc[2:].copy()
dados_brutos.columns = cabecalho_original
dados_brutos = dados_brutos.dropna(how="all").reset_index(drop=True)

print("Dimensão bruta:", dados_brutos.shape)
print("Colunas originais:")
print(list(dados_brutos.columns))
display(dados_brutos.head())

Dimensão bruta: (944, 15)
Colunas originais:
['NÚMERO DO PROCESSO', 'DT. DISTRIBUIÇÃO', 'SORTEIO', 'HÍBRIDA', 'TEVE ACORDO NA AC?', 'TÉCNICA', 'TEVE ACORDO FORA DA AC?', 'SENTENÇA', 'DT. SENTENÇA', 'DURAÇÃO-SENT2', 'DURAÇÃO-TECN', 'DURAÇÃO-SORT', 'DURAÇÃO-PROC', 'DURAÇÃO DO PROCESSO', 'STATUS']


1,NÚMERO DO PROCESSO,DT. DISTRIBUIÇÃO,SORTEIO,HÍBRIDA,TEVE ACORDO NA AC?,TÉCNICA,TEVE ACORDO FORA DA AC?,SENTENÇA,DT. SENTENÇA,DURAÇÃO-SENT2,DURAÇÃO-TECN,DURAÇÃO-SORT,DURAÇÃO-PROC,DURAÇÃO DO PROCESSO,STATUS
0,0800001-35.2026.8.20.5004,2026-01-01 00:00:00,Rito Atual Ac Opcional,Não,Não,**,Não,Extinção do feito sem julgamento de mérito,2026-01-07 00:00:00,6,NaN,6,6,6 dia(s),Encerrado
1,0800024-78.2026.8.20.5004,2026-01-06 00:00:00,Rito Atual Ac Opcional,Não,Não,**,NaN,NaN,NaN,274,NaN,274,274,274 dia(s),Ativo
2,0808292-43.2025.8.20.5300,2026-01-06 00:00:00,Rito Atual Ac Opcional,Não,Não,**,Não,Extinção do Feito sem Julgamento de Mérito,2026-01-09 00:00:00,3,NaN,3,3,3 dia(s),Encerrado
3,0800038-62.2026.8.20.5004,2026-01-07 00:00:00,Ac Obrigatória Presencial,Não,Sim,Desconstrução da Verdade,Não,Extinção do Feito com Homologação do Acordo,2026-02-22 00:00:00,46,46,46,46,46 dia(s),Encerrado
4,0800035-10.2026.8.20.5004,2026-01-07 00:00:00,Rito Atual Ac Opcional,Não,Não,**,Não,Extinção do Feito com Julgamento de Mérito,2026-02-27 00:00:00,51,NaN,51,51,51 dia(s),Encerrado


## 4. Limpeza dos Dados


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **padronização de rótulos e limpeza semântica**.

> **Atenção e interpretação:** ao analisar os resultados, compare os nomes originais e padronizados para garantir consistência terminológica.

In [4]:
# Passo a passo da celula:
# 1) Declarar funcoes auxiliares para organizar a logica.
# 2) Padronizar e limpar rotulos/valores com regras de qualidade.
# 3) Converter tipos para formato analitico consistente.
# 4) Exibir saidas intermediarias para validacao da etapa.

# Correcao de rotulos com mojibake recorrente e padronizacao em snake_case

CORRECOES_COLUNA = {
    "N\ufffdMERO DO PROCESSO": "NUMERO DO PROCESSO",
    "NMERO DO PROCESSO": "NUMERO DO PROCESSO",
    "DT. DISTRIBUI\ufffd\ufffdO": "DT. DISTRIBUICAO",
    "DT. DISTRIBUIO": "DT. DISTRIBUICAO",
    "H\ufffdBRIDA": "HIBRIDA",
    "HBRIDA": "HIBRIDA",
    "T\ufffdCNICA": "TECNICA",
    "TCNICA": "TECNICA",
    "SENTEN\ufffdA": "SENTENCA",
    "SENTENA": "SENTENCA",
    "DT. SENTEN\ufffdA": "DT. SENTENCA",
    "DT. SENTENA": "DT. SENTENCA",
    "DURA\ufffd\ufffdO-SENT2": "DURACAO-SENT2",
    "DURAO-SENT2": "DURACAO-SENT2",
    "DURA\ufffd\ufffdO-TECN": "DURACAO-TECN",
    "DURAO-TECN": "DURACAO-TECN",
    "DURA\ufffd\ufffdO-SORT": "DURACAO-SORT",
    "DURAO-SORT": "DURACAO-SORT",
    "DURA\ufffd\ufffdO-PROC": "DURACAO-PROC",
    "DURAO-PROC": "DURACAO-PROC",
    " DURA\ufffd\ufffdO DO PROCESSO": "DURACAO DO PROCESSO",
    " DURAO DO PROCESSO": "DURACAO DO PROCESSO",
}


def slug_coluna(nome: str) -> str:
    nome = CORRECOES_COLUNA.get(nome, nome)
    nome = unicodedata.normalize("NFKD", str(nome)).encode("ascii", "ignore").decode("ascii")
    nome = nome.lower().strip()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    nome = re.sub(r"_+", "_", nome).strip("_")
    return nome


mapa_renomear = {c: slug_coluna(c) for c in dados_brutos.columns}
dados = dados_brutos.rename(columns=mapa_renomear).copy()

for col in dados.select_dtypes(include="object").columns:
    dados[col] = (
        dados[col]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({"nan": np.nan, "": np.nan})
    )


display(pd.DataFrame({"original": list(mapa_renomear.keys()), "padronizado": list(mapa_renomear.values())}))
print("Colunas padronizadas:", list(dados.columns))

,original,padronizado
0,NÚMERO DO PROCESSO,numero_do_processo
1,DT. DISTRIBUIÇÃO,dt_distribuicao
2,SORTEIO,sorteio
3,HÍBRIDA,hibrida
4,TEVE ACORDO NA AC?,teve_acordo_na_ac
5,TÉCNICA,tecnica
6,TEVE ACORDO FORA DA AC?,teve_acordo_fora_da_ac
7,SENTENÇA,sentenca
8,DT. SENTENÇA,dt_sentenca
9,DURAÇÃO-SENT2,duracao_sent2


Colunas padronizadas: ['numero_do_processo', 'dt_distribuicao', 'sorteio', 'hibrida', 'teve_acordo_na_ac', 'tecnica', 'teve_acordo_fora_da_ac', 'sentenca', 'dt_sentenca', 'duracao_sent2', 'duracao_tecn', 'duracao_sort', 'duracao_proc', 'duracao_do_processo', 'status']


## 5. Normalização das Categorias: Partições


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **normalização das categorias analíticas**.

> **Atenção e interpretação:** ao analisar os resultados, valide se os grupos experimentais foram mapeados corretamente, sem categorias residuais.

In [5]:
# Passo a passo da celula:
# 1) Padronizar e limpar rotulos/valores com regras de qualidade.
# 2) Exibir saidas intermediarias para validacao da etapa.

# Harmonização de categorias-chave para reduzir variações de digitação/grafia
if "status" in dados.columns:
    mapa_status = {
        "encerrado": "Encerrado",
        "ativo": "Ativo",
    }
    dados["status_norm"] = (
        dados["status"].str.lower().map(mapa_status).fillna(dados["status"])
    )

if "sorteio" in dados.columns:
    # Partições experimentais esperadas no desenho do estudo
    dados["grupo_experimento"] = (
        dados["sorteio"]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace({
            "Rito Atual Ac Opcional": "Controle",
            "Ac Obrigatória Presencial": "Tratamento_presencial",
            "Ac Remota": "Tratamento_remoto",
        })
    )

if "hibrida" in dados.columns:
    dados["hibrida_norm"] = dados["hibrida"].replace({"Não": "Nao", "Sim": "Sim"})

display(dados[[c for c in ["sorteio", "grupo_experimento", "status", "status_norm"] if c in dados.columns]].head(10))

1,sorteio,grupo_experimento,status,status_norm
0,Rito Atual Ac Opcional,Controle,Encerrado,Encerrado
1,Rito Atual Ac Opcional,Controle,Ativo,Ativo
2,Rito Atual Ac Opcional,Controle,Encerrado,Encerrado
3,Ac Obrigatória Presencial,Tratamento_presencial,Encerrado,Encerrado
4,Rito Atual Ac Opcional,Controle,Encerrado,Encerrado
5,Ac Remota,Tratamento_remoto,Encerrado,Encerrado
6,Ac Remota,Tratamento_remoto,Encerrado,Encerrado
7,Rito Atual Ac Opcional,Controle,Encerrado,Encerrado
8,Rito Atual Ac Opcional,Controle,Encerrado,Encerrado
9,Ac Remota,Tratamento_remoto,Encerrado,Encerrado


## 6. Tipagem Final do Dataset


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **tipagem final das variáveis**.

> **Atenção e interpretação:** ao analisar os resultados, verifique se datas e durações ficaram nos tipos esperados para as análises subsequentes.

In [6]:
# Passo a passo da celula:
# 1) Converter tipos para formato analitico consistente.
# 2) Gerar estatisticas descritivas e tabelas de inspecao.
# 3) Exibir saidas intermediarias para validacao da etapa.

# Conversões de tipo para datas e variáveis numéricas de duração
COLS_DATA = [c for c in ["dt_distribuicao", "dt_sentenca"] if c in dados.columns]
for c in COLS_DATA:
    dados[c] = pd.to_datetime(dados[c], errors="coerce")

COLS_NUM = [
    c for c in [
        "duracao_sent2", "duracao_tecn", "duracao_sort", "duracao_proc"
    ] if c in dados.columns
]
for c in COLS_NUM:
    dados[c] = pd.to_numeric(dados[c], errors="coerce")

if "duracao_do_processo" in dados.columns:
    dados["duracao_do_processo_txt"] = dados["duracao_do_processo"]

display(dados[COLS_DATA + COLS_NUM].describe(include="all").T if (COLS_DATA or COLS_NUM) else dados.head())
print("dtypes finais:")
print(dados.dtypes)

,count,mean,min,25%,50%,75%,max,std
1,,,,,,,,
dt_distribuicao,944,2026-04-24 00:19:49.830508,2026-01-01 00:00:00,2026-03-05 18:00:00,2026-04-26 12:00:00,2026-06-12 00:00:00,2026-08-03 00:00:00,NaN
dt_sentenca,734,2026-06-16 10:00:19.618528,2026-01-07 00:00:00,2026-04-29 00:00:00,2026-06-22 00:00:00,2026-08-10 00:00:00,2026-10-06 00:00:00,NaN
duracao_sent2,943.0,78.457052,0.0,42.0,80.0,110.5,274.0,51.206256
duracao_tecn,428.0,105.806075,1.0,79.0,98.5,126.25,257.0,36.105327
duracao_sort,943.0,78.457052,0.0,42.0,80.0,110.5,274.0,51.206256
duracao_proc,943.0,78.457052,0.0,42.0,80.0,110.5,274.0,51.206256


dtypes finais:
1
numero_do_processo                    str
dt_distribuicao            datetime64[us]
sorteio                               str
hibrida                               str
teve_acordo_na_ac                     str
tecnica                               str
teve_acordo_fora_da_ac                str
sentenca                              str
dt_sentenca                datetime64[us]
duracao_sent2                     float64
duracao_tecn                      float64
duracao_sort                      float64
duracao_proc                      float64
duracao_do_processo                   str
status                                str
status_norm                           str
grupo_experimento                     str
hibrida_norm                          str
duracao_do_processo_txt               str
dtype: object


## 7. Auditoria de Qualidade e Coerência


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **auditoria de qualidade e coerência**.

> **Atenção e interpretação:** ao analisar os resultados, priorize inconsistências de datas, duração e chaves duplicadas para eventual correção.

In [7]:
# Passo a passo da celula:
# 1) Executar rotina principal desta celula.

# Regras de consistência análogas ao modelo: duplicatas, faltantes e coerência temporal
relatorio = {}

# 1) Duplicidade de processo
if "numero_do_processo" in dados.columns:
    relatorio["duplicatas_numero_processo"] = int(dados["numero_do_processo"].duplicated().sum())

# 2) Faltantes nas variaveis principais
for c in ["dt_distribuicao", "dt_sentenca", "duracao_proc", "status_norm", "grupo_experimento"]:
    if c in dados.columns:
        relatorio[f"faltantes_{c}"] = int(dados[c].isna().sum())

# 3) Duracao negativa (incoerencia)
if "duracao_proc" in dados.columns:
    relatorio["duracao_proc_negativa"] = int((dados["duracao_proc"] < 0).sum())

# 4) Checagem cruzada entre datas e duracao
if {"dt_distribuicao", "dt_sentenca", "duracao_proc"}.issubset(dados.columns):
    dias_data = (dados["dt_sentenca"] - dados["dt_distribuicao"]).dt.days
    dif = (dias_data - dados["duracao_proc"]).abs()
    relatorio["inconsistencia_data_x_duracao_maior_2_dias"] = int((dif > 2).sum(skipna=True))

pd.Series(relatorio, name="valor")

duplicatas_numero_processo                      6
faltantes_dt_distribuicao                       0
faltantes_dt_sentenca                         210
faltantes_duracao_proc                          1
faltantes_status_norm                           1
faltantes_grupo_experimento                     0
duracao_proc_negativa                           0
inconsistencia_data_x_duracao_maior_2_dias      0
Name: valor, dtype: int64

## 8. Exportação do Dataset Limpo


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **exportação de artefatos**.

> **Atenção e interpretação:** ao analisar os resultados, confirme o caminho do arquivo salvo e utilize-o como base reprodutível das etapas seguintes.

In [8]:
# Passo a passo da celula:
# 1) Definir caminhos de entrada e saida usados no estudo.
# 2) Exportar artefatos finais para reuso e auditoria.
# 3) Exibir saidas intermediarias para validacao da etapa.

# Persistência do dataset limpo para reutilização em outros notebooks/relatórios
saida_limpa = SAIDA_DIR / "estcaso_4jec_natal_limpo.xlsx"
dados.to_excel(saida_limpa, index=False)
print(f"Arquivo salvo: {saida_limpa}")

Arquivo salvo: e:\DADOS\Documents\AmbienteDeTrabalho\Projetos\notebook\dat\processado\estcaso_4jec_natal_limpo.xlsx


## 9. Auditoria Complementar: Prescrição, Censura e Regras Textuais


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **janela observacional e truncamento**.

> **Atenção e interpretação:** ao analisar os resultados, verifique se a janela de observação pode enviesar as estimativas de duração.

In [ ]:
# Passo a passo da celula:
# 1) Executar rotina principal desta celula.

# Indicadores de desenho experimental, com apoio do ODT de limitações
resumo_metodologico = {}

# Censura ? direita aproximada: processos ativos ainda sem sentença final
if "status_norm" in dados.columns:
    resumo_metodologico["proporcao_ativos"] = float((dados["status_norm"] == "Ativo").mean())

# Janela de observação relatada no ODT (~218 dias)
if "duracao_proc" in dados.columns:
    resumo_metodologico["acima_218_dias"] = float((dados["duracao_proc"] > 218).mean())

# Balanceamento entre grupos experimentais
if "grupo_experimento" in dados.columns:
    resumo_metodologico["n_grupos"] = int(dados["grupo_experimento"].nunique(dropna=True))

# Verificação textual simples das limitações explicitadas no ODT
odt_norm = normalizar_texto_curto(texto_odt).lower()
for termo in ["validade externa", "censura", "right-censoring", "homogeneidade", "hawthorne"]:
    resumo_metodologico[f"odt_menciona_{termo}"] = termo in odt_norm

pd.Series(resumo_metodologico, name="valor")

## 10. Análise de Variáveis Categóricas (Exercício 1)


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **análise de variáveis categóricas**.

> **Atenção e interpretação:** ao analisar os resultados, interprete concentração, cauda longa de categorias e possíveis assimetrias de distribuição.

In [ ]:
# Passo a passo da celula:
# 1) Declarar funcoes auxiliares para organizar a logica.
# 2) Converter tipos para formato analitico consistente.
# 3) Gerar estatisticas descritivas e tabelas de inspecao.
# 4) Produzir graficos para visualizacao dos padroes.
# 5) Exibir saidas intermediarias para validacao da etapa.

# Função de barras + Pareto para variáveis categóricas

def plot_barras_pareto(df: pd.DataFrame, coluna: str, top_n: int = 12):
    freq = df[coluna].value_counts(dropna=False).head(top_n)
    prop = freq / freq.sum() * 100
    acum = prop.cumsum()

    fig, ax1 = plt.subplots(figsize=(12, 5))
    bars = ax1.bar(freq.index.astype(str), freq.values, color="#4C72B0", alpha=0.85)
    ax1.set_ylabel("Frequência")
    ax1.set_xlabel(coluna)
    ax1.tick_params(axis="x", rotation=45)

    for b, v in zip(bars, freq.values):
        ax1.text(b.get_x() + b.get_width() / 2, v, str(int(v)), ha="center", va="bottom", fontsize=9)

    ax2 = ax1.twinx()
    ax2.plot(freq.index.astype(str), acum.values, color="#DD8452", marker="o")
    ax2.set_ylabel("% acumulado")
    ax2.set_ylim(0, 105)

    plt.title(f"Distribuição de {coluna} (barras + Pareto)")
    plt.tight_layout()
    plt.show()

for cat_col in [c for c in ["grupo_experimento", "status_norm", "teve_acordo_na_ac", "teve_acordo_fora_da_ac", "sentenca"] if c in dados.columns]:
    print("
", "=" * 90)
    print(cat_col)
    print(dados[cat_col].value_counts(dropna=False).head(12))
    plot_barras_pareto(dados, cat_col)

## 11. Análise de Variáveis Quantitativas (Exercício 2)


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **análise descritiva de variáveis quantitativas**.

> **Atenção e interpretação:** ao analisar os resultados, observe mediana, dispersão, assimetria e valores extremos para orientar a escolha de testes.

In [ ]:
# Passo a passo da celula:
# 1) Gerar estatisticas descritivas e tabelas de inspecao.
# 2) Produzir graficos para visualizacao dos padroes.
# 3) Exibir saidas intermediarias para validacao da etapa.

# Estatística descritiva e gráficos de distribuição
quants = [c for c in ["duracao_sent2", "duracao_tecn", "duracao_sort", "duracao_proc"] if c in dados.columns]

descr = dados[quants].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).T
display(descr)

for c in quants:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    sns.histplot(dados[c].dropna(), kde=True, ax=axes[0], color="#55A868")
    axes[0].set_title(f"Histograma: {c}")

    sns.boxplot(x=dados[c], ax=axes[1], color="#C44E52")
    axes[1].set_title(f"Boxplot: {c}")

    plt.tight_layout()
    plt.show()

## 12. Comparação de Variáveis Quantitativas por Grupos (Exercício 3)


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **análise descritiva de variáveis quantitativas**.

> **Atenção e interpretação:** ao analisar os resultados, observe mediana, dispersão, assimetria e valores extremos para orientar a escolha de testes.

In [ ]:
# Passo a passo da celula:
# 1) Produzir graficos para visualizacao dos padroes.
# 2) Executar testes estatisticos para avaliar hipoteses.
# 3) Exibir saidas intermediarias para validacao da etapa.

# Comparação não paramétrica entre grupos do experimento
if {"grupo_experimento", "duracao_proc"}.issubset(dados.columns):
    grupos = [g["duracao_proc"].dropna().values for _, g in dados.dropna(subset=["grupo_experimento"]).groupby("grupo_experimento")]
    nomes = list(dados.dropna(subset=["grupo_experimento"]).groupby("grupo_experimento").groups.keys())

    if len(grupos) >= 2:
        if len(grupos) == 2:
            u, p = stats.mannwhitneyu(grupos[0], grupos[1], alternative="two-sided")
            print("Mann-Whitney U")
            print(f"Grupos: {nomes[0]} vs {nomes[1]} | U={u:.2f} | p={p:.4g}")
        else:
            h, p = stats.kruskal(*grupos)
            print("Kruskal-Wallis")
            print(f"Grupos: {nomes} | H={h:.2f} | p={p:.4g}")

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=dados, x="grupo_experimento", y="duracao_proc")
    sns.stripplot(data=dados, x="grupo_experimento", y="duracao_proc", alpha=0.25, color="black")
    plt.xticks(rotation=20)
    plt.title("Duração do processo por grupo experimental")
    plt.tight_layout()
    plt.show()

## 13. Análise de Associações entre Variáveis Categóricas (Exercício 4)


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **associação entre variáveis categóricas**.

> **Atenção e interpretação:** ao analisar os resultados, avalie se h? dependência entre categorias e em quais combinações o padrão ? mais forte.

In [ ]:
# Passo a passo da celula:
# 1) Gerar estatisticas descritivas e tabelas de inspecao.
# 2) Produzir graficos para visualizacao dos padroes.
# 3) Executar testes estatisticos para avaliar hipoteses.
# 4) Exibir saidas intermediarias para validacao da etapa.

# Qui-quadrado entre grupo e status; mosaico se statsmodels estiver disponível
if {"grupo_experimento", "status_norm"}.issubset(dados.columns):
    tab = pd.crosstab(dados["grupo_experimento"], dados["status_norm"], dropna=False)
    display(tab)

    chi2, p, gl, esp = stats.chi2_contingency(tab)
    print(f"Qui-quadrado: chi2={chi2:.3f}, gl={gl}, p={p:.4g}")

    if mosaic is not None:
        plt.figure(figsize=(8, 5))
        mosaic(tab.stack())
        plt.title("Mosaico: grupo_experimento ? status_norm")
        plt.show()

## 14. Análise de Relações entre Variáveis Quantitativas (Exercício 5)


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **relações entre variáveis quantitativas**.

> **Atenção e interpretação:** ao analisar os resultados, considere direção e intensidade das relações, distinguindo correlação de causalidade.

In [ ]:
# Passo a passo da celula:
# 1) Produzir graficos para visualizacao dos padroes.
# 2) Exibir saidas intermediarias para validacao da etapa.

# Correlação de Spearman para variáveis de duração
quants = [c for c in ["duracao_sent2", "duracao_tecn", "duracao_sort", "duracao_proc"] if c in dados.columns]
if len(quants) >= 2:
    corr_spear = dados[quants].corr(method="spearman")
    display(corr_spear)

    plt.figure(figsize=(7, 5))
    sns.heatmap(corr_spear, annot=True, fmt=".2f", cmap="RdBu_r", center=0)
    plt.title("Correlação de Spearman")
    plt.tight_layout()
    plt.show()

if {"duracao_sort", "duracao_proc"}.issubset(dados.columns):
    plt.figure(figsize=(8, 5))
    sns.regplot(data=dados, x="duracao_sort", y="duracao_proc", scatter_kws={"alpha": 0.4})
    plt.title("Duração sorteio vs duração total")
    plt.tight_layout()
    plt.show()

## 15. Tempo de Tramitação: Definição e Descrição


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **análise descritiva de variáveis quantitativas**.

> **Atenção e interpretação:** ao analisar os resultados, observe mediana, dispersão, assimetria e valores extremos para orientar a escolha de testes.

In [ ]:
# Passo a passo da celula:
# 1) Gerar estatisticas descritivas e tabelas de inspecao.
# 2) Produzir graficos para visualizacao dos padroes.
# 3) Exibir saidas intermediarias para validacao da etapa.

# Definição da variável-alvo principal
if "duracao_proc" in dados.columns:
    dados["tempo_tramitacao_dias"] = dados["duracao_proc"]

alvo = "tempo_tramitacao_dias" if "tempo_tramitacao_dias" in dados.columns else None
if alvo:
    print(dados[alvo].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]))

    plt.figure(figsize=(10, 4))
    sns.histplot(dados[alvo].dropna(), bins=35, kde=True)
    plt.title("Distribuição do tempo de tramitação (dias)")
    plt.show()

## 16. O Tempo Segue Distribuição Normal?


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **avaliação de normalidade**.

> **Atenção e interpretação:** ao analisar os resultados, combine testes formais e inspeção visual para decidir entre abordagens paramétricas e robustas.

In [ ]:
# Passo a passo da celula:
# 1) Produzir graficos para visualizacao dos padroes.
# 2) Executar testes estatisticos para avaliar hipoteses.
# 3) Exibir saidas intermediarias para validacao da etapa.

# Testes de normalidade (amostras muito grandes podem rejeitar por pequenas assimetrias)
if alvo:
    serie = dados[alvo].dropna()
    amostra = serie.sample(min(len(serie), 500), random_state=42)

    shapiro_stat, shapiro_p = stats.shapiro(amostra)
    dag_stat, dag_p = stats.normaltest(amostra)

    print(f"Shapiro-Wilk: W={shapiro_stat:.4f}, p={shapiro_p:.4g}")
    print(f"D'Agostino-Pearson: K2={dag_stat:.4f}, p={dag_p:.4g}")

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111)
    stats.probplot(amostra, dist="norm", plot=ax)
    ax.set_title("Q-Q plot da variável de tempo")
    plt.tight_layout()
    plt.show()

## 17. Testes de Hipóteses: do Simples ao Completo


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **comparação não paramétrica entre grupos**.

> **Atenção e interpretação:** ao analisar os resultados, interprete o valor-p junto com os gráficos, evitando conclusões apenas pela significância.

In [ ]:
# Passo a passo da celula:
# 1) Executar testes estatisticos para avaliar hipoteses.
# 2) Exibir saidas intermediarias para validacao da etapa.

# Exemplo 1: proporção de encerramento (teste binomial aproximado via z)
if "status_norm" in dados.columns:
    n = dados["status_norm"].notna().sum()
    k = (dados["status_norm"] == "Encerrado").sum()
    p0 = 0.5
    p_hat = k / n if n else np.nan
    se = np.sqrt(p0 * (1 - p0) / n) if n else np.nan
    z = (p_hat - p0) / se if n else np.nan
    p_val = 2 * (1 - stats.norm.cdf(abs(z))) if n else np.nan

    print(f"n={n}, encerrados={k}, p_hat={p_hat:.3f}, z={z:.3f}, p={p_val:.4g}")

# Exemplo 2: comparação de tempo entre encerrados e ativos
if {"status_norm", "duracao_proc"}.issubset(dados.columns):
    a = dados.loc[dados["status_norm"] == "Encerrado", "duracao_proc"].dropna()
    b = dados.loc[dados["status_norm"] == "Ativo", "duracao_proc"].dropna()
    if len(a) > 0 and len(b) > 0:
        u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        print(f"Mann-Whitney (Encerrado vs Ativo): U={u:.2f}, p={p:.4g}")

## 18. Qual Distribuição Descreve o Tempo?


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **ajuste de distribuições probabilísticas**.

> **Atenção e interpretação:** ao analisar os resultados, compare AIC/BIC e prefira modelos parcimoniosos com bom ajuste global.

In [ ]:
# Passo a passo da celula:
# 1) Executar testes estatisticos para avaliar hipoteses.
# 2) Ajustar modelos para quantificar relacoes entre variaveis.
# 3) Exibir saidas intermediarias para validacao da etapa.

# Ajustes paramétricos simples e comparação por AIC
if alvo:
    x = dados[alvo].dropna().values
    x = x[np.isfinite(x)]
    x = x[x > 0]

    candidatos = {
        "lognorm": stats.lognorm,
        "gamma": stats.gamma,
        "weibull_min": stats.weibull_min,
        "expon": stats.expon,
    }

    resultados = []
    for nome, dist in candidatos.items():
        params = dist.fit(x)
        loglik = np.sum(dist.logpdf(x, *params))
        k = len(params)
        aic = 2 * k - 2 * loglik
        bic = np.log(len(x)) * k - 2 * loglik
        resultados.append((nome, aic, bic, params))

    tabela_fit = pd.DataFrame(resultados, columns=["dist", "AIC", "BIC", "params"]).sort_values("AIC")
    display(tabela_fit)

## 19. Modelos de Regressão: Tempo, Acordo e Encerramento


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **modelagem de regressão**.

> **Atenção e interpretação:** ao analisar os resultados, interprete coeficientes, direção de efeito e significância, sempre com leitura substantiva do contexto.

In [ ]:
# Passo a passo da celula:
# 1) Converter tipos para formato analitico consistente.
# 2) Ajustar modelos para quantificar relacoes entre variaveis.
# 3) Exibir saidas intermediarias para validacao da etapa.

# Regressão linear (tempo) e logística (encerrado) com variáveis observáveis
if sm is not None:
    if {"duracao_proc", "duracao_sort", "duracao_sent2"}.issubset(dados.columns):
        reg_df = dados[["duracao_proc", "duracao_sort", "duracao_sent2"]].dropna().copy()
        if len(reg_df) > 20:
            y = reg_df["duracao_proc"]
            X = sm.add_constant(reg_df[["duracao_sort", "duracao_sent2"]])
            ols = sm.OLS(y, X).fit()
            print(ols.summary())

    if {"status_norm", "duracao_sort", "duracao_sent2"}.issubset(dados.columns):
        log_df = dados[["status_norm", "duracao_sort", "duracao_sent2"]].dropna().copy()
        log_df["status_bin"] = (log_df["status_norm"] == "Encerrado").astype(int)
        if log_df["status_bin"].nunique() == 2 and len(log_df) > 30:
            X = sm.add_constant(log_df[["duracao_sort", "duracao_sent2"]])
            model = sm.Logit(log_df["status_bin"], X).fit(disp=False)
            print(model.summary())

## 20. Análise de Sobrevivência: Inclusão de Procedimentos Não Concluídos


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **análise de sobrevivência**.

> **Atenção e interpretação:** ao analisar os resultados, observe diferenças entre curvas e o impacto da censura na interpretação temporal.

In [ ]:
# Passo a passo da celula:
# 1) Converter tipos para formato analitico consistente.
# 2) Produzir graficos para visualizacao dos padroes.
# 3) Ajustar modelos para quantificar relacoes entre variaveis.
# 4) Aplicar analise de sobrevivencia com dados censurados.

# Kaplan-Meier para tempo até encerramento
if KaplanMeierFitter is not None and {"duracao_proc", "status_norm"}.issubset(dados.columns):
    km_df = dados[["duracao_proc", "status_norm", "grupo_experimento"]].dropna(subset=["duracao_proc", "status_norm"]).copy()
    km_df["evento"] = (km_df["status_norm"] == "Encerrado").astype(int)

    plt.figure(figsize=(10, 6))
    for grupo, sub in km_df.groupby("grupo_experimento", dropna=False):
        kmf = KaplanMeierFitter()
        kmf.fit(durations=sub["duracao_proc"], event_observed=sub["evento"], label=str(grupo))
        kmf.plot_survival_function(ci_show=False)

    plt.title("Kaplan-Meier por grupo experimental")
    plt.ylabel("Probabilidade de permanecer em tramitação")
    plt.xlabel("Dias")
    plt.tight_layout()
    plt.show()

## 21. Tópico Avançado: Janela de Observação e Truncamento


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **janela observacional e truncamento**.

> **Atenção e interpretação:** ao analisar os resultados, verifique se a janela de observação pode enviesar as estimativas de duração.

In [ ]:
# Passo a passo da celula:
# 1) Gerar estatisticas descritivas e tabelas de inspecao.
# 2) Produzir graficos para visualizacao dos padroes.
# 3) Exibir saidas intermediarias para validacao da etapa.

# Avaliação da janela observacional e possível truncamento temporal
if {"dt_distribuicao", "duracao_proc"}.issubset(dados.columns):
    corte_inicio = dados["dt_distribuicao"].min()
    corte_fim = dados["dt_distribuicao"].max()
    janela = (corte_fim - corte_inicio).days if pd.notna(corte_inicio) and pd.notna(corte_fim) else np.nan

    print(f"Período de distribuição observado: {corte_inicio} a {corte_fim} (~{janela} dias)")
    print("Percentis da duração:")
    print(dados["duracao_proc"].describe(percentiles=[0.5, 0.75, 0.9, 0.95]))

    plt.figure(figsize=(10, 4))
    sns.ecdfplot(data=dados, x="duracao_proc")
    plt.axvline(218, color="red", linestyle="--", label="janela de 218 dias")
    plt.legend()
    plt.title("ECDF da duração com marca da janela de observação")
    plt.tight_layout()
    plt.show()

## 22. Complementação Metodológica (Escala Ampliada)


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **etapa analítica do pipeline**.

> **Atenção e interpretação:** ao analisar os resultados, compare a saída com os objetivos da seção e registre eventuais desvios para revisão.

In [ ]:
# Passo a passo da celula:
# 1) Definir caminhos de entrada e saida usados no estudo.

# Espaço para acoplar bases externas (ex.: DataJud/TJRN) mantendo o mesmo pipeline
# Comentário metodológico: este bloco permanece preparado para integração futura.

checklist_extensao = {
    "1_chave_de_linkage_definida": "numero_do_processo" in dados.columns,
    "2_variavel_tempo_harmonizada": "duracao_proc" in dados.columns,
    "3_status_evento_harmonizado": "status_norm" in dados.columns,
    "4_grupos_experimentais_harmonizados": "grupo_experimento" in dados.columns,
}

pd.Series(checklist_extensao, name="pronto_para_ampliacao")

## 23. Exportação Final e Conclusões


**Frase-guia didática.** Nesta etapa, executaremos a rotina de **exportação de artefatos**.

> **Atenção e interpretação:** ao analisar os resultados, confirme o caminho do arquivo salvo e utilize-o como base reprodutível das etapas seguintes.

In [ ]:
# Passo a passo da celula:
# 1) Definir caminhos de entrada e saida usados no estudo.
# 2) Exportar artefatos finais para reuso e auditoria.
# 3) Exibir saidas intermediarias para validacao da etapa.

# Síntese automatizada de achados essenciais
sintese = {}
if "duracao_proc" in dados.columns:
    sintese["n_processos"] = int(dados["duracao_proc"].notna().sum())
    sintese["mediana_dias"] = float(dados["duracao_proc"].median())
    sintese["p90_dias"] = float(dados["duracao_proc"].quantile(0.9))

if "status_norm" in dados.columns:
    sintese["prop_encerrados"] = float((dados["status_norm"] == "Encerrado").mean())

if "grupo_experimento" in dados.columns:
    sintese["grupos_detectados"] = int(dados["grupo_experimento"].nunique(dropna=True))

print("Resumo do estudo de caso:")
print(pd.Series(sintese))

saida_csv = SAIDA_DIR / "estcaso_4jec_natal_limpo.csv"
dados.to_csv(saida_csv, index=False, encoding="utf-8-sig")
print(f"
CSV exportado: {saida_csv}")

## Referências

- Documento empírico sobre conciliação no 4º JEC de Natal (DOCX);
- Documento sobre limitações metodológicas do experimento (ODT);
- Estrutura analítica inspirada no pipeline de `rafaelcc_ed1.ipynb`.
